# 2D-to-3D Game Models — Real AI Pipeline on Colab

Converts a 2D image to a textured 3D model (.GLB) using **TripoSG** (SOTA geometry, MIT license, ~8GB VRAM).

**How to use:** Runtime → Change runtime type → **T4 GPU** → **Run All**

1. First Run All: installs everything + auto-restarts
2. Second Run All: skips install → upload image → generates 3D → download .GLB

**Best input images:** Single object, background removed or white, centered, 3/4 view angle, 512x512+ PNG

In [ ]:
#@title 1. Setup — install TripoSG + pipeline (auto-skips on 2nd run)
import os, sys, subprocess

REPO_DIR = '/content/2d-to-3d-game-models'
TRIPOSG_DIR = '/content/TripoSG'
INSTALL_MARKER = '/content/.deps_installed_v2'

if not os.path.exists(INSTALL_MARKER):
    print('=== Installing packages + TripoSG (~5 min) ===')
    os.chdir('/content')

    # Clone our pipeline repo
    subprocess.run(['rm', '-rf', REPO_DIR])
    subprocess.check_call(['git', 'clone', '-b',
        'claude/image-to-3d-pipeline-CnSII',
        'https://github.com/pmikola/2d-to-3d-game-models.git'])

    # Clone TripoSG
    if not os.path.exists(TRIPOSG_DIR):
        subprocess.check_call(['git', 'clone',
            'https://github.com/VAST-AI-Research/TripoSG.git'])

    # Install scipy compatible with Colab numpy 2.x
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'scipy', 'onnxruntime-gpu'])

    # rembg for background removal (pinned for Colab compat)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--no-deps', 'rembg==2.0.57'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pooch', 'pymatting', 'scikit-image', 'filetype', 'imagehash'])

    # TripoSG dependencies
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'ninja'])  # needed for diso compilation
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'diso', '--no-build-isolation'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'diffusers', 'transformers', 'einops', 'trimesh', 'omegaconf',
        'peft', 'jaxtyping', 'typeguard', 'pymeshlab',
        'pygltflib', 'xatlas', 'Pillow', 'accelerate', 'safetensors',
        'pyyaml', 'tqdm', 'huggingface_hub'])

    open(INSTALL_MARKER, 'w').write('done')
    print('\n=== Done! Restarting kernel... ===')
    print('>>> Click Run All again after restart <<<\n')
    try:
        import IPython
        IPython.get_ipython().kernel.do_shutdown(True)
    except Exception:
        os._exit(0)

else:
    print('=== Packages ready, skipping install ===')
    os.chdir(REPO_DIR)

    import numpy, trimesh, torch
    print(f'numpy={numpy.__version__}  trimesh={trimesh.__version__}')

    REMBG_OK = False
    try:
        from rembg import remove
        REMBG_OK = True
        print('rembg: OK')
    except Exception:
        print('rembg: unavailable — will skip background removal')

    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
    print('\n=== Ready! ===')

In [ ]:
#@title 2. Upload your image (tap Choose Files)
from google.colab import files
from PIL import Image
import numpy as np
from IPython.display import display

INPUT_PATH = '/content/test_input.png'

print('Upload a PNG or JPG (single object, clean background, centered):')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    import shutil
    shutil.copy(fname, INPUT_PATH)
    img = Image.open(INPUT_PATH)
    print(f'\nUploaded: {fname} ({img.size[0]}x{img.size[1]})')
    display(img.resize((300, 300)))
else:
    print('No upload — using synthetic test image')
    arr = np.zeros((400, 400, 3), dtype=np.uint8)
    arr[:] = 200
    y, x = np.ogrid[-200:200, -200:200]
    arr[x**2 + y**2 < 120**2] = [220, 140, 60]
    Image.fromarray(arr).save(INPUT_PATH)

In [ ]:
#@title 3. Generate 3D model (TripoSG) → Repair → UV → PBR → GLB
import os, sys, time, shutil, tempfile, base64
import torch
import trimesh
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML
from huggingface_hub import snapshot_download

import logging
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')

os.chdir('/content/2d-to-3d-game-models')

from pipeline.mesh_repair import repair_and_prepare
from pipeline.geometry import normalize_mesh, unwrap_uvs, save_mesh_as_obj
from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps
from pipeline.export import export_textured_dir_to_glb, validate_glb

INPUT_PATH = '/content/test_input.png'
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_GLB = str(OUTPUT_DIR / 'model.glb')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32

start = time.time()

# === Stage 1: Load TripoSG models ===
print('\n[1/7] Loading TripoSG models (first run downloads ~3GB)...')
TRIPOSG_DIR = '/content/TripoSG'
sys.path.insert(0, TRIPOSG_DIR)
sys.path.insert(0, os.path.join(TRIPOSG_DIR, 'scripts'))

weights_dir = os.path.join(TRIPOSG_DIR, 'pretrained_weights')
os.makedirs(weights_dir, exist_ok=True)

triposg_weights = os.path.join(weights_dir, 'TripoSG')
rmbg_weights = os.path.join(weights_dir, 'RMBG-1.4')

if not os.path.exists(os.path.join(triposg_weights, 'config.json')):
    print('  Downloading TripoSG weights...')
    snapshot_download('VAST-AI/TripoSG', local_dir=triposg_weights)
if not os.path.exists(os.path.join(rmbg_weights, 'config.json')):
    print('  Downloading RMBG-1.4 weights...')
    snapshot_download('briaai/RMBG-1.4', local_dir=rmbg_weights)

from triposg.pipelines.pipeline_triposg import TripoSGPipeline
from scripts.briarmbg import BriaRMBG
from scripts.image_process import prepare_image

rmbg_net = BriaRMBG.from_pretrained(rmbg_weights).to(device)
rmbg_net.eval()
pipe = TripoSGPipeline.from_pretrained(triposg_weights).to(device, dtype)
print(f'  TripoSG loaded on {device}')

# === Stage 2: Preprocess image ===
print('\n[2/7] Preprocessing image (background removal)...')
# prepare_image takes a FILE PATH (string), not a PIL Image
img_processed = prepare_image(INPUT_PATH, bg_color=np.array([1.0, 1.0, 1.0]), rmbg_net=rmbg_net)
print(f'  Preprocessed: {img_processed.size}')
display(img_processed.resize((200, 200)))

# Keep original for texturing later
image_pil = Image.open(INPUT_PATH).convert('RGB')

# === Stage 3: Generate 3D mesh with TripoSG ===
print('\n[3/7] Generating 3D geometry with TripoSG (~2-5 min on T4)...')
with torch.no_grad():
    outputs = pipe(
        image=img_processed,
        generator=torch.Generator(device=device).manual_seed(42),
        num_inference_steps=50,
        guidance_scale=7.0,
    ).samples[0]

mesh = trimesh.Trimesh(
    vertices=outputs[0].astype(np.float32),
    faces=np.ascontiguousarray(outputs[1])
)
print(f'  Generated: {len(mesh.vertices)} vertices, {len(mesh.faces)} faces')

# Free GPU memory
del pipe, rmbg_net
torch.cuda.empty_cache()

# === Stage 4: Mesh repair ===
print('\n[4/7] Repairing mesh (topology, smoothing, watertight)...')
mesh = repair_and_prepare(mesh, smooth_iterations=5)
print(f'  After repair: {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

# === Stage 5: Normalize + UV unwrap ===
print('\n[5/7] UV unwrapping (xatlas)...')
normalize_mesh(mesh)
unwrap_uvs(mesh)
print(f'  UV unwrapped: {len(mesh.vertices)} verts')

# === Stage 6: PBR texture maps ===
print('\n[6/7] Generating PBR maps from input image...')
texture_img = image_pil.resize((1024, 1024))

with tempfile.TemporaryDirectory() as tmp:
    obj_path = save_mesh_as_obj(mesh, tmp)
    textured = Path(tmp) / 'textured'
    textured.mkdir()

    texture_img.save(str(textured / 'texture_atlas.png'))
    shutil.copy(obj_path, str(textured / 'mesh_textured.obj'))

    pbr = generate_pbr_maps(texture_img, strength=1.5)
    save_pbr_maps(pbr, str(textured / 'pbr'))

    row = Image.new('RGB', (256*4, 256))
    row.paste(texture_img.resize((256,256)), (0,0))
    row.paste(pbr['normal'].resize((256,256)), (256,0))
    row.paste(pbr['roughness'].convert('RGB').resize((256,256)), (512,0))
    row.paste(pbr['metallic'].convert('RGB').resize((256,256)), (768,0))
    print('  Albedo | Normal | Roughness | Metallic:')
    display(row)

    # === Stage 7: Export GLB ===
    print('\n[7/7] Exporting GLB with embedded textures...')
    export_textured_dir_to_glb(str(textured), OUTPUT_GLB)

# Validate
info = validate_glb(OUTPUT_GLB)
elapsed = time.time() - start
size_mb = os.path.getsize(OUTPUT_GLB) / (1024*1024)

print(f'\n{"="*60}')
print(f'DONE in {elapsed:.1f}s')
print(f'  GLB: {OUTPUT_GLB} ({size_mb:.2f} MB)')
print(f'  Vertices: {info.get("total_vertices", "?")}')
print(f'  Faces: {info.get("total_faces", "?")}')
print(f'  Valid: {info.get("valid", False)}')
print(f'{"="*60}')

# === DOWNLOAD ===
print('\n')
with open(OUTPUT_GLB, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()
display(HTML(
    f'<h2><a href="data:model/gltf-binary;base64,{b64}" '
    f'download="model.glb" '
    f'style="background:#4CAF50;color:white;padding:15px 30px;'
    f'text-decoration:none;border-radius:8px;font-size:18px;">'
    f'📥 TAP HERE TO DOWNLOAD model.glb ({size_mb:.1f} MB)</a></h2>'
))
print('View your model: https://gltf-viewer.donmccurdy.com/')